In [13]:
import json
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
import matplotlib.pyplot as plt
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")
np.random.seed(42)
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
import pkg_resources

# List installed packages and versions

pkgs = sorted([(d.project_name, d.version) for d in pkg_resources.working_set], key=lambda x: x[0].lower())
for name, ver in pkgs:
    print(f"{name}=={ver}")

In [2]:
import pandas as pd

if 'train_df' not in globals():
    train_df = pd.read_parquet("train_preprocessed.parquet")
train_with_cluster = train_df.copy()

for cid in [0, 1]:
    mask = train_with_cluster["Demand_Cluster"] == cid
    demand = train_with_cluster.loc[mask, "Order_Demand"]
    zero_rate = (demand == 0).mean() * 100
    print(f"\nCluster {cid}:")
    print(f"  Rows       : {mask.sum():,}")
    print(f"  Zero rate  : {zero_rate:.1f}%")
    print(f"  Mean       : {demand.mean():.1f}")
    print(f"  Median     : {demand.median():.1f}")
    print(f"  P90        : {demand.quantile(0.9):.1f}")
    print(f"  P99        : {demand.quantile(0.99):.1f}")
    print(f"  Max        : {demand.max():.1f}")


Cluster 0:
  Rows       : 151,363
  Zero rate  : 50.4%
  Mean       : 2253.5
  Median     : 0.0
  P90        : 2000.0
  P99        : 40000.0
  Max        : 2500000.0

Cluster 1:
  Rows       : 89,051
  Zero rate  : 61.5%
  Mean       : 768.2
  Median     : 0.0
  P90        : 462.0
  P99        : 15000.0
  Max        : 837000.0


In [3]:
def log_transform(y):
    return np.log1p(np.maximum(y, 0))


def inverse_log_transform(y_log):
    return np.maximum(np.expm1(y_log), 0)


def compute_metrics(y_true, y_pred, label=""):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    nonzero = y_true != 0
    mape = (
        np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])) * 100
        if nonzero.any() else np.nan
    )
    wape = (
        np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100
        if y_true.sum() != 0 else np.nan
    )
    prefix = f"[{label}] " if label else ""
    print(f"    {prefix}RMSE={rmse:.2f}  MAE={mae:.2f}  MAPE={mape:.1f}%  WAPE={wape:.1f}%")
    return {"rmse": rmse, "mae": mae, "mape": mape, "wape": wape}

In [5]:

import lightgbm as lgb
import optuna
import matplotlib.pyplot as plt
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore")
np.random.seed(42)
optuna.logging.set_verbosity(optuna.logging.WARNING)

def log_transform(y):
    return np.log1p(np.maximum(y, 0))


def inverse_log_transform(y_log):
    return np.maximum(np.expm1(y_log), 0)


def compute_metrics(y_true, y_pred, label=""):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    nonzero = y_true != 0
    mape = (
        np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])) * 100
        if nonzero.any() else np.nan
    )
    wape = (
        np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100
        if y_true.sum() != 0 else np.nan
    )
    prefix = f"[{label}] " if label else ""
    print(f"    {prefix}RMSE={rmse:.2f}  MAE={mae:.2f}  MAPE={mape:.1f}%  WAPE={wape:.1f}%")
    return {"rmse": rmse, "mae": mae, "mape": mape, "wape": wape}


def wape_score(y_true, y_pred):
    """WAPE thấp = tốt. Dùng làm hàm mục tiêu Optuna."""
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    denom = np.sum(np.abs(y_true))
    return np.sum(np.abs(y_true - y_pred)) / denom * 100 if denom > 0 else np.nan

def add_temporal_features(df):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["Day"] = df["Date"].dt.day
    df["DayOfWeek"] = df["Date"].dt.dayofweek
    df["DayOfYear"] = df["Date"].dt.dayofyear
    df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
    df["Is_Weekend"] = df["DayOfWeek"].isin([5, 6]).astype(int)
    return df


def add_lag_features(df, group_cols):
    df = df.sort_values(by=group_cols + ["Date"]).reset_index(drop=True)
    for lag in [7, 14, 30]:
        df[f"Demand_Lag_{lag}"] = df.groupby(group_cols)["Order_Demand"].shift(lag)
    for window in [7, 30]:
        df[f"Demand_Rolling_Mean_{window}"] = (
            df.groupby(group_cols)["Order_Demand"]
            .shift(1)
            .transform(lambda x: x.rolling(window, min_periods=1).mean())
        )
    fill_cols = [c for c in df.columns if "Lag_" in c or "Rolling_" in c]
    df[fill_cols] = df[fill_cols].fillna(0)
    return df


def add_advanced_intermittent_features(df, group_cols):
    df = df.sort_values(by=group_cols + ["Date"]).reset_index(drop=True)
    shifted = df.groupby(group_cols)["Order_Demand"].shift(1)
    for window in [7, 30]:
        df[f"Zero_Rate_{window}d"] = (
            shifted.transform(lambda x: (x == 0).rolling(window, min_periods=1).mean())
            .fillna(0.5)
        )

    def days_since_last_sale(series_dates, series_demand):
        result = np.zeros(len(series_demand), dtype=float)
        last_sale_date = None
        for i, (d, v) in enumerate(zip(series_dates, series_demand)):
            if last_sale_date is None:
                result[i] = -1
            else:
                result[i] = int((d - last_sale_date) / np.timedelta64(1, 'D'))
            if v > 0:
                last_sale_date = d
        return result

    df["Days_Since_Last_Sale"] = (
        df.groupby(group_cols, group_keys=False)
        .apply(lambda g: pd.Series(
            days_since_last_sale(g["Date"].values, g["Order_Demand"].values),
            index=g.index,
        ))
    )

    df["Demand_Velocity_7d"] = (
        df.groupby(group_cols)["Order_Demand"]
        .shift(1)
        .transform(lambda x: x.diff().abs().rolling(7, min_periods=1).mean())
        .fillna(0)
    )

    def consecutive_zeros(series):
        result, count = np.zeros(len(series)), 0
        for i, val in enumerate(series):
            result[i] = count
            count = count + 1 if val == 0 else 0
        return result

    df["Consecutive_Zeros"] = df.groupby(group_cols)["Order_Demand"].transform(
        lambda x: consecutive_zeros(x.values)
    )

    return df

def out_of_fold_target_encoding(train_df, val_df, test_df, col, target_col, global_mean, n_splits=5):

    enc_col = f"{col}_TE"
    train_df = train_df.copy()
    val_df = val_df.copy()
    test_df = test_df.copy()

    oof_enc = np.full(len(train_df), global_mean)
    kf = KFold(n_splits=n_splits, shuffle=False)
    for fold_idx, (trn_idx, oof_idx) in enumerate(kf.split(train_df)):
        fold_mean = (
            train_df.iloc[trn_idx]
            .groupby(col)[target_col]
            .mean()
        )
        oof_enc[oof_idx] = train_df.iloc[oof_idx][col].map(fold_mean).fillna(global_mean).values
    train_df[enc_col] = oof_enc

    full_mean = train_df.groupby(col)[target_col].mean()
    val_df[enc_col] = val_df[col].map(full_mean).fillna(global_mean)
    test_df[enc_col] = test_df[col].map(full_mean).fillna(global_mean)

    return train_df, val_df, test_df

train_df = pd.read_parquet("train_preprocessed.parquet")
val_df = pd.read_parquet("val_preprocessed.parquet")
test_df = pd.read_parquet("test_preprocessed.parquet")

with open("pipeline_meta.json") as f:
    meta = json.load(f)

FEATURE_COLS = meta["feature_cols"]
global_mean = meta["global_mean"]
TARGET_COL = "Order_Demand"
GROUP_KEYS = ["Product_Code", "Warehouse"]

for df in [train_df, val_df, test_df]:
    df["Date"] = pd.to_datetime(df["Date"])

train_end = train_df["Date"].max()
val_end = val_df["Date"].max()

full_df = (
    pd.concat([train_df, val_df, test_df])
    .sort_values(by=GROUP_KEYS + ["Date"])
    .reset_index(drop=True)
)
full_df = add_temporal_features(full_df)
full_df = add_lag_features(full_df, GROUP_KEYS)
full_df = add_advanced_intermittent_features(full_df, GROUP_KEYS)

train_df = full_df[full_df["Date"] <= train_end].copy()
val_df = full_df[(full_df["Date"] > train_end) & (full_df["Date"] <= val_end)].copy()
test_df = full_df[full_df["Date"] > val_end].copy()

new_cols = [
    c for c in full_df.columns
    if c not in FEATURE_COLS + [TARGET_COL, "Date"] + GROUP_KEYS
]
FEATURE_COLS += new_cols

for col in ["Product_Code", "Warehouse", "Demand_Cluster"]:
    if col in train_df.columns:
        train_df, val_df, test_df = out_of_fold_target_encoding(
            train_df, val_df, test_df, col, TARGET_COL, global_mean
        )
        enc_col = f"{col}_TE"
        if enc_col not in FEATURE_COLS:
            FEATURE_COLS.append(enc_col)
        print(f"  {col} → {enc_col} done")

MODEL_FEATURES = [c for c in FEATURE_COLS if c not in ["Demand_Cluster", "Product_Code", "Warehouse"]]

MODEL_FEATURES = [c for c in MODEL_FEATURES if c in train_df.columns]

X_train, y_train = train_df[MODEL_FEATURES], train_df[TARGET_COL]
X_val, y_val = val_df[MODEL_FEATURES], val_df[TARGET_COL]
X_test, y_test = test_df[MODEL_FEATURES], test_df[TARGET_COL]

print(f"\n  Train : {len(train_df):,} | {train_df['Date'].min().date()} -> {train_end.date()}")
print(f"  Val   : {len(val_df):,} | {val_df['Date'].min().date()} -> {val_end.date()}")
print(f"  Test  : {len(test_df):,} | {test_df['Date'].min().date()} -> {test_df['Date'].max().date()}")
print(f"  Features: {len(MODEL_FEATURES)} | Zero demand: {(y_train == 0).mean() * 100:.1f}%")


# Model 1: Baseline (Log Transform) 
print("Model 1 — Baseline: Single LightGBM + Log Transform")
print("=" * 65)

baseline = lgb.LGBMRegressor(
    n_estimators=1000, learning_rate=0.05, random_state=42, n_jobs=-1, verbosity=-1
)
baseline.fit(
    X_train, log_transform(y_train.values),
    eval_set=[(X_val, log_transform(y_val.values))],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)
preds_baseline = inverse_log_transform(baseline.predict(X_test))
metrics_baseline = compute_metrics(y_test, preds_baseline, label="Baseline")


# Model 2: Tweedie Default 
print("Model 2 — Tweedie Loss (p=1.5, default)")
print("=" * 65)

tweedie_model = lgb.LGBMRegressor(
    objective="tweedie",
    tweedie_variance_power=1.5,
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)
tweedie_model.fit(
    X_train, y_train.values,
    eval_set=[(X_val, y_val.values)],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)
preds_tweedie = np.maximum(tweedie_model.predict(X_test), 0)
metrics_tweedie = compute_metrics(y_test, preds_tweedie, label="Tweedie (p=1.5)")


# Model 3: Hybrid Calibrated 
print("Model 3 — Hybrid Calibrated (base, soft-blend)")

raw_clf = lgb.LGBMClassifier(
    n_estimators=1000, learning_rate=0.05, max_depth=6, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbosity=-1,
)
raw_clf.fit(
    X_train, (y_train > 0).astype(int),
    eval_set=[(X_val, (y_val > 0).astype(int))],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)

calibrated_clf = CalibratedClassifierCV(raw_clf, cv="prefit", method="isotonic")
calibrated_clf.fit(X_val, (y_val > 0).astype(int))

raw_probs_val = raw_clf.predict_proba(X_val)[:, 1]
cal_probs_val = calibrated_clf.predict_proba(X_val)[:, 1]
print(f"  Raw clf     - mean prob: {raw_probs_val.mean():.3f}  std: {raw_probs_val.std():.3f}")
print(f"  Calibrated  - mean prob: {cal_probs_val.mean():.3f}  std: {cal_probs_val.std():.3f}")
print(f"  Actual non-zero rate (val): {(y_val > 0).mean():.3f}")

nonzero_tr = y_train > 0
nonzero_val = y_val > 0

stage2_reg = lgb.LGBMRegressor(
    n_estimators=1000, learning_rate=0.05, max_depth=6, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbosity=-1,
)
stage2_reg.fit(
    X_train[nonzero_tr], log_transform(y_train[nonzero_tr].values),
    eval_set=[(X_val[nonzero_val], log_transform(y_val[nonzero_val].values))],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)

cal_probs_test = calibrated_clf.predict_proba(X_test)[:, 1]
raw_demand_test = inverse_log_transform(stage2_reg.predict(X_test))
preds_hybrid_soft = cal_probs_test * raw_demand_test
metrics_hybrid_soft = compute_metrics(y_test, preds_hybrid_soft, label="Hybrid (Soft-blend)")


cal_probs_val_test = calibrated_clf.predict_proba(X_val)[:, 1]
raw_demand_val = inverse_log_transform(stage2_reg.predict(X_val))

best_threshold, best_wape_val = 0.0, float("inf")
thresholds = np.arange(0.0, 0.55, 0.05)

for thr in thresholds:
    preds_val_hard = np.where(cal_probs_val_test < thr, 0.0, raw_demand_val)
    w = wape_score(y_val.values, preds_val_hard)
    print(f"    threshold={thr:.2f}  Val WAPE={w:.2f}%")
    if w < best_wape_val:
        best_wape_val = w
        best_threshold = thr

print(f"\n  Best threshold = {best_threshold:.2f}  (Val WAPE = {best_wape_val:.2f}%)")

preds_hybrid_hard = np.where(cal_probs_test < best_threshold, 0.0, raw_demand_test)
metrics_hybrid_hard = compute_metrics(y_test, preds_hybrid_hard, label=f"Hybrid (Hard thr={best_threshold:.2f})")


def objective_hybrid(trial):
    clf_params = dict(
        n_estimators=1000,
        learning_rate=trial.suggest_float("clf_lr", 0.01, 0.1, log=True),
        max_depth=trial.suggest_int("clf_max_depth", 3, 6),       # MAX 6
        num_leaves=trial.suggest_int("clf_num_leaves", 15, 63),   # MAX 63
        min_child_samples=trial.suggest_int("clf_min_child", 20, 100),
        subsample=trial.suggest_float("clf_subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("clf_colsample", 0.6, 1.0),
        reg_alpha=trial.suggest_float("clf_reg_alpha", 1e-3, 5.0, log=True),
        reg_lambda=trial.suggest_float("clf_reg_lambda", 1e-3, 5.0, log=True),
        random_state=42, n_jobs=-1, verbosity=-1,
        is_unbalance=True,  
    )
    reg_params = dict(
        n_estimators=1000,
        learning_rate=trial.suggest_float("reg_lr", 0.01, 0.1, log=True),
        max_depth=trial.suggest_int("reg_max_depth", 3, 6),
        num_leaves=trial.suggest_int("reg_num_leaves", 15, 63),
        min_child_samples=trial.suggest_int("reg_min_child", 20, 100),
        subsample=trial.suggest_float("reg_subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("reg_colsample", 0.6, 1.0),
        reg_alpha=trial.suggest_float("reg_reg_alpha", 1e-3, 5.0, log=True),
        reg_lambda=trial.suggest_float("reg_reg_lambda", 1e-3, 5.0, log=True),
        random_state=42, n_jobs=-1, verbosity=-1,
    )
    threshold = trial.suggest_float("threshold", 0.0, 0.5)

    # Classifier
    clf = lgb.LGBMClassifier(**clf_params)
    clf.fit(
        X_train, (y_train > 0).astype(int),
        eval_set=[(X_val, (y_val > 0).astype(int))],
        callbacks=[lgb.early_stopping(30, verbose=False)],
    )
    cal = CalibratedClassifierCV(clf, cv="prefit", method="isotonic")
    cal.fit(X_val, (y_val > 0).astype(int))

    # Regressor 
    reg = lgb.LGBMRegressor(**reg_params)
    reg.fit(
        X_train[nonzero_tr], log_transform(y_train[nonzero_tr].values),
        eval_set=[(X_val[nonzero_val], log_transform(y_val[nonzero_val].values))],
        callbacks=[lgb.early_stopping(30, verbose=False)],
    )

    probs_v = cal.predict_proba(X_val)[:, 1]
    demand_v = inverse_log_transform(reg.predict(X_val))
    preds_v = np.where(probs_v < threshold, 0.0, demand_v)
    return wape_score(y_val.values, preds_v)


study_hybrid = optuna.create_study(direction="minimize")
study_hybrid.optimize(objective_hybrid, n_trials=50, show_progress_bar=True)
print(f"\nBest WAPE (Val): {study_hybrid.best_value:.2f}%")
print(f"Best params: {json.dumps(study_hybrid.best_params, indent=2)}")


# Model 4: Hybrid Tuned (Optuna) 
print("Model 4 — Hybrid Tuned (Optuna)")


bp = study_hybrid.best_params
best_threshold_optuna = bp.pop("threshold", best_threshold)

clf_tuned = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=bp["clf_lr"],
    max_depth=bp["clf_max_depth"],
    num_leaves=bp["clf_num_leaves"],
    min_child_samples=bp["clf_min_child"],
    subsample=bp["clf_subsample"],
    colsample_bytree=bp["clf_colsample"],
    reg_alpha=bp["clf_reg_alpha"],
    reg_lambda=bp["clf_reg_lambda"],
    is_unbalance=True,
    random_state=42, n_jobs=-1, verbosity=-1,
)
clf_tuned.fit(
    X_train, (y_train > 0).astype(int),
    eval_set=[(X_val, (y_val > 0).astype(int))],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)
cal_tuned = CalibratedClassifierCV(clf_tuned, cv="prefit", method="isotonic")
cal_tuned.fit(X_val, (y_val > 0).astype(int))

reg_tuned = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=bp["reg_lr"],
    max_depth=bp["reg_max_depth"],
    num_leaves=bp["reg_num_leaves"],
    min_child_samples=bp["reg_min_child"],
    subsample=bp["reg_subsample"],
    colsample_bytree=bp["reg_colsample"],
    reg_alpha=bp["reg_reg_alpha"],
    reg_lambda=bp["reg_reg_lambda"],
    random_state=42, n_jobs=-1, verbosity=-1,
)
reg_tuned.fit(
    X_train[nonzero_tr], log_transform(y_train[nonzero_tr].values),
    eval_set=[(X_val[nonzero_val], log_transform(y_val[nonzero_val].values))],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)

probs_tuned_test = cal_tuned.predict_proba(X_test)[:, 1]
demand_tuned_test = inverse_log_transform(reg_tuned.predict(X_test))
preds_hybrid_tuned = np.where(probs_tuned_test < best_threshold_optuna, 0.0, demand_tuned_test)
metrics_hybrid_tuned = compute_metrics(y_test, preds_hybrid_tuned, label="Hybrid Tuned")


preds_tweedie_val = np.maximum(tweedie_model.predict(X_val), 0)

probs_tuned_val = cal_tuned.predict_proba(X_val)[:, 1]
demand_tuned_val = inverse_log_transform(reg_tuned.predict(X_val))
preds_hybrid_tuned_val = np.where(probs_tuned_val < best_threshold_optuna, 0.0, demand_tuned_val)

best_w, best_ensemble_wape = 0.5, float("inf")
for w in np.arange(0.0, 1.05, 0.05):
    blend_val = w * preds_hybrid_tuned_val + (1 - w) * preds_tweedie_val
    w_score = wape_score(y_val.values, blend_val)
    if w_score < best_ensemble_wape:
        best_ensemble_wape = w_score
        best_w = w

print(f" Best weight: Hybrid={best_w:.2f}  Tweedie={1-best_w:.2f}  (Val WAPE = {best_ensemble_wape:.2f}%)")

preds_ensemble = best_w * preds_hybrid_tuned + (1 - best_w) * preds_tweedie
metrics_ensemble = compute_metrics(y_test, preds_ensemble, label=f"Ensemble (w_hybrid={best_w:.2f})")

print("Feature Importance — Hybrid Stage 2 Regressor Tuned (Top 25)")


importance_df = (
    pd.DataFrame({"feature": MODEL_FEATURES, "gain": reg_tuned.feature_importances_})
    .sort_values("gain", ascending=False)
    .reset_index(drop=True)
)
print(importance_df.head(25).to_string(index=False))


# Visualization 
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 1. Feature importance
top25 = importance_df.head(25)
axes[0, 0].barh(top25["feature"][::-1], top25["gain"][::-1], color="steelblue")
axes[0, 0].set_title("Top 25 Feature Importance (Gain) — Hybrid Regressor Tuned")
axes[0, 0].set_xlabel("Gain")

# 2. Probability calibration
axes[0, 1].hist(raw_probs_val, bins=50, alpha=0.6, label="Raw", color="tomato")
axes[0, 1].hist(cal_probs_val, bins=50, alpha=0.6, label="Calibrated (base)", color="seagreen")
axes[0, 1].axvline(x=(y_val > 0).mean(), color="black", linestyle="--", label="Actual non-zero rate")
axes[0, 1].set_title("Classifier Probability Distribution (Val)")
axes[0, 1].set_xlabel("P(demand > 0)")
axes[0, 1].legend()

# 3. Threshold search curve
val_wapes = []
for thr in thresholds:
    pv = np.where(cal_probs_val_test < thr, 0.0, raw_demand_val)
    val_wapes.append(wape_score(y_val.values, pv))
axes[1, 0].plot(thresholds, val_wapes, marker="o", color="steelblue")
axes[1, 0].axvline(x=best_threshold, color="red", linestyle="--", label=f"Best={best_threshold:.2f}")
axes[1, 0].set_title("Hard-Threshold Search on Val")
axes[1, 0].set_xlabel("Threshold")
axes[1, 0].set_ylabel("WAPE (%)")
axes[1, 0].legend()

# 4. WAPE comparison
all_results_ordered = [
    ("Baseline\n(Log)", metrics_baseline["wape"]),
    ("Tweedie\n(p=1.5)", metrics_tweedie["wape"]),
    ("Hybrid\n(Soft-v2)", metrics_hybrid_soft["wape"]),
    ("Hybrid\n(Hard-thr)", metrics_hybrid_hard["wape"]),
    ("Hybrid\n(Tuned)", metrics_hybrid_tuned["wape"]),
    ("Ensemble\n(Weighted)", metrics_ensemble["wape"]),
]
model_names_viz = [x[0] for x in all_results_ordered]
wape_vals_viz = [x[1] for x in all_results_ordered]
colors_viz = ["gray", "steelblue", "seagreen", "mediumseagreen", "darkgreen", "gold"]
bars = axes[1, 1].bar(model_names_viz, wape_vals_viz, color=colors_viz)
axes[1, 1].set_title("WAPE Comparison — All Models (Test Set)")
axes[1, 1].set_ylabel("WAPE (%)")
for bar, val in zip(bars, wape_vals_viz):
    axes[1, 1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3, f"{val:.1f}%", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("analysis_v3.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n  Saved: analysis_v3.png")

print("FINAL COMPARISON — ALL MODELS (Test Set)")


all_results = {
    "Baseline (Log Transform)":       metrics_baseline,
    "Tweedie (p=1.5, default)":       metrics_tweedie,
    "Hybrid (Soft-blend, base)":      metrics_hybrid_soft,
    f"Hybrid (Hard thr={best_threshold:.2f})": metrics_hybrid_hard,
    "Hybrid Tuned (Optuna)":          metrics_hybrid_tuned,
    f"Ensemble (w_hybrid={best_w:.2f})": metrics_ensemble,
}

base_wape = metrics_baseline["wape"]
print(f"\n  {'Model':<40s} {'RMSE':>10} {'MAE':>10} {'WAPE':>10} {'vs Baseline':>14}")
print(f"  {'-' * 86}")
for name, m in all_results.items():
    delta = (m["wape"] - base_wape) / base_wape * 100
    tag = f"{'↓' if delta < 0 else '↑'} {abs(delta):.1f}%" if name != "Baseline (Log Transform)" else "(base)"
    print(f"  {name:<40s} {m['rmse']:>10.2f} {m['mae']:>10.2f} {m['wape']:>9.2f}%  {tag:>14}")

print("\n  Notes:")
print(f"  - Hard threshold optimized: {best_threshold:.2f} (on Val)")
print(f"  - Ensemble weight optimized: Hybrid={best_w:.2f}, Tweedie={1-best_w:.2f}")

Building features on continuous timeline...
  Product_Code → Product_Code_TE done
  Warehouse → Warehouse_TE done
  Demand_Cluster → Demand_Cluster_TE done

  Train : 240,414 | 2011-01-10 -> 2015-06-01
  Val   : 57,744 | 2015-06-08 -> 2016-03-21
  Test  : 53,998 | 2016-03-28 -> 2017-01-09
  Features: 65 | Zero demand: 54.5%
Model 1 — Baseline: Single LightGBM + Log Transform
    [Baseline] RMSE=22532.54  MAE=1624.52  MAPE=155.4%  WAPE=77.7%
Model 2 — Tweedie Loss (p=1.5, default)
    [Tweedie (p=1.5)] RMSE=20682.03  MAE=1744.71  MAPE=407.9%  WAPE=83.4%
Model 3 — Hybrid Calibrated (base, soft-blend)
  Raw clf     - mean prob: 0.471  std: 0.499
  Calibrated  - mean prob: 0.471  std: 0.499
  Actual non-zero rate (val): 0.471
    [Hybrid (Soft-blend)] RMSE=22238.16  MAE=1611.56  MAPE=149.0%  WAPE=77.0%
    threshold=0.00  Val WAPE=110.89%
    threshold=0.05  Val WAPE=77.98%
    threshold=0.10  Val WAPE=77.98%
    threshold=0.15  Val WAPE=77.98%
    threshold=0.20  Val WAPE=77.98%
    thres

Best trial: 31. Best value: 74.9763: 100%|██████████| 50/50 [11:18<00:00, 13.57s/it]



Best WAPE (Val): 74.98%
Best params: {
  "clf_lr": 0.03694015961739043,
  "clf_max_depth": 4,
  "clf_num_leaves": 28,
  "clf_min_child": 47,
  "clf_subsample": 0.7237065453272216,
  "clf_colsample": 0.9357626752107598,
  "clf_reg_alpha": 0.004576696108969492,
  "clf_reg_lambda": 0.002964012259353159,
  "reg_lr": 0.020307141053527305,
  "reg_max_depth": 4,
  "reg_num_leaves": 43,
  "reg_min_child": 20,
  "reg_subsample": 0.9276085361618281,
  "reg_colsample": 0.8754435901201286,
  "reg_reg_alpha": 0.012024461268138118,
  "reg_reg_lambda": 0.004326572987978986,
  "threshold": 0.0734340234995807
}
Model 4 — Hybrid Tuned (Optuna)
    [Hybrid Tuned] RMSE=20043.25  MAE=1518.60  MAPE=149.3%  WAPE=72.6%
 Best weight: Hybrid=0.90  Tweedie=0.10  (Val WAPE = 74.96%)
    [Ensemble (w_hybrid=0.90)] RMSE=19976.46  MAE=1518.83  MAPE=173.2%  WAPE=72.6%
Feature Importance — Hybrid Stage 2 Regressor Tuned (Top 25)
               feature  gain
Demand_Rolling_Mean_30  1233
 Demand_Expanding_Mean   963
  

In [6]:
import pickle
with open("hybrid_model.pkl", "wb") as f:
    pickle.dump({
        "clf_tuned": clf_tuned,
        "cal_tuned": cal_tuned,
        "reg_tuned": reg_tuned,
        "threshold": best_threshold_optuna,
        "model_features": MODEL_FEATURES,
    }, f)

In [7]:
MODEL_FEATURES_NO_CLUSTER = [c for c in MODEL_FEATURES if c != "Demand_Cluster_TE"]
print(f"  Full features     : {len(MODEL_FEATURES)}")
print(f"  Without cluster   : {len(MODEL_FEATURES_NO_CLUSTER)}")

X_train_noclu = X_train[MODEL_FEATURES_NO_CLUSTER]
X_val_noclu   = X_val[MODEL_FEATURES_NO_CLUSTER]
X_test_noclu  = X_test[MODEL_FEATURES_NO_CLUSTER]

clf_noclu = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=bp["clf_lr"], max_depth=bp["clf_max_depth"], num_leaves=bp["clf_num_leaves"],
    min_child_samples=bp["clf_min_child"], subsample=bp["clf_subsample"], colsample_bytree=bp["clf_colsample"],
    reg_alpha=bp["clf_reg_alpha"], reg_lambda=bp["clf_reg_lambda"],
    is_unbalance=True, random_state=42, n_jobs=-1, verbosity=-1,
)
clf_noclu.fit(
    X_train_noclu, (y_train > 0).astype(int),
    eval_set=[(X_val_noclu, (y_val > 0).astype(int))],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)
cal_noclu = CalibratedClassifierCV(clf_noclu, cv="prefit", method="isotonic")
cal_noclu.fit(X_val_noclu, (y_val > 0).astype(int))

reg_noclu = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=bp["reg_lr"], max_depth=bp["reg_max_depth"], num_leaves=bp["reg_num_leaves"],
    min_child_samples=bp["reg_min_child"], subsample=bp["reg_subsample"], colsample_bytree=bp["reg_colsample"],
    reg_alpha=bp["reg_reg_alpha"], reg_lambda=bp["reg_reg_lambda"],
    random_state=42, n_jobs=-1, verbosity=-1,
)
reg_noclu.fit(
    X_train_noclu[nonzero_tr], log_transform(y_train[nonzero_tr].values),
    eval_set=[(X_val_noclu[nonzero_val], log_transform(y_val[nonzero_val].values))],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)

probs_noclu_test = cal_noclu.predict_proba(X_test_noclu)[:, 1]
demand_noclu_test = inverse_log_transform(reg_noclu.predict(X_test_noclu))
preds_noclu = np.where(probs_noclu_test < best_threshold_optuna, 0.0, demand_noclu_test)
metrics_noclu = compute_metrics(y_test, preds_noclu, label="Hybrid (No Cluster)")

print(f"\n  {'Model':<35s} {'#Features':>10} {'RMSE':>10} {'MAE':>10} {'WAPE':>10}")
print(f"  {'-' * 76}")
print(f"  {'Hybrid Tuned (With Cluster)':<35s} {len(MODEL_FEATURES):>10} "
      f"{metrics_hybrid_tuned['rmse']:>10.2f} {metrics_hybrid_tuned['mae']:>10.2f} {metrics_hybrid_tuned['wape']:>9.2f}%")
print(f"  {'Hybrid Tuned (Without Cluster)':<35s} {len(MODEL_FEATURES_NO_CLUSTER):>10} "
      f"{metrics_noclu['rmse']:>10.2f} {metrics_noclu['mae']:>10.2f} {metrics_noclu['wape']:>9.2f}%")

delta_wape_cluster = metrics_noclu["wape"] - metrics_hybrid_tuned["wape"]
print(f"\n  Delta WAPE (No-Cluster - With-Cluster): {delta_wape_cluster:+.2f}pp")
if delta_wape_cluster > 0:
    print("  -> Removing the cluster feature HURTS WAPE: cluster carries causal predictive signal.")
elif delta_wape_cluster < 0:
    print("  -> Removing the cluster feature IMPROVES WAPE: cluster may be redundant given other features.")
else:
    print("  -> No measurable difference.")

with open("cluster_ablation.json", "w") as f:
    json.dump({
        "wape_with_cluster": metrics_hybrid_tuned["wape"],
        "wape_without_cluster": metrics_noclu["wape"],
        "rmse_with_cluster": metrics_hybrid_tuned["rmse"],
        "rmse_without_cluster": metrics_noclu["rmse"],
        "mae_with_cluster": metrics_hybrid_tuned["mae"],
        "mae_without_cluster": metrics_noclu["mae"],
        "delta_wape_pp": delta_wape_cluster,
    }, f, indent=2)


  Full features     : 65
  Without cluster   : 64
    [Hybrid (No Cluster)] RMSE=21074.79  MAE=1556.43  MAPE=152.1%  WAPE=74.4%

  Model                                #Features       RMSE        MAE       WAPE
  ----------------------------------------------------------------------------
  Hybrid Tuned (With Cluster)                 65   20043.25    1518.60     72.60%
  Hybrid Tuned (Without Cluster)              64   21074.79    1556.43     74.41%

  Delta WAPE (No-Cluster - With-Cluster): +1.81pp
  -> Removing the cluster feature HURTS WAPE: cluster carries causal predictive signal.


# Post-training validation script

In [11]:
import lightgbm as lgb
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")
np.random.seed(42)

def compute_metrics_full(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    nz   = y_true != 0
    mape = np.mean(np.abs((y_true[nz] - y_pred[nz]) / y_true[nz])) * 100 if nz.any() else np.nan
    wape = np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100 if y_true.sum() != 0 else np.nan
    zero_pred_rate = (y_pred == 0).mean() * 100
    return dict(rmse=rmse, mae=mae, mape=mape, wape=wape, zero_pred_rate=zero_pred_rate)


def wape_score(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    denom = np.sum(np.abs(y_true))
    return np.sum(np.abs(y_true - y_pred)) / denom * 100 if denom > 0 else np.nan

THRESHOLD_OPTUNA = best_threshold_optuna

cal_probs_val = cal_tuned.predict_proba(X_val)[:, 1]
demand_val_pred = inverse_log_transform(reg_tuned.predict(X_val))
 
candidate_thresholds = np.arange(0.0, 0.55, 0.05)
val_wapes = []
for t in candidate_thresholds:
    preds_t = np.where(cal_probs_val < t, 0.0, demand_val_pred)
    val_wapes.append(wape_score(y_val, preds_t))
 
THRESHOLD_VAL = float(candidate_thresholds[int(np.argmin(val_wapes))])
print(f"  THRESHOLD_OPTUNA (from saved model) : {THRESHOLD_OPTUNA:.4f}")
print(f"  THRESHOLD_VAL    (recomputed sweep) : {THRESHOLD_VAL:.4f}  (Val WAPE={min(val_wapes):.2f}%)")

# Predictions on Test with Hybrid Tuned
probs_tuned_test = cal_tuned.predict_proba(X_test)[:, 1]
demand_tuned_test = inverse_log_transform(reg_tuned.predict(X_test))

# Apply threshold from Optuna
preds_val_optuna_thr  = np.where(probs_tuned_val  < THRESHOLD_OPTUNA, 0.0, demand_tuned_val)
preds_test_optuna_thr = np.where(probs_tuned_test < THRESHOLD_OPTUNA, 0.0, demand_tuned_test)

m_val_opt  = compute_metrics_full(y_val,  preds_val_optuna_thr)
m_test_opt = compute_metrics_full(y_test, preds_test_optuna_thr)

gap_wape = m_test_opt["wape"] - m_val_opt["wape"]

print(f"\n  [Threshold = {THRESHOLD_OPTUNA}]")
print(f"  Val  WAPE = {m_val_opt['wape']:.2f}%  (zero_pred={m_val_opt['zero_pred_rate']:.1f}%)")
print(f"  Test WAPE = {m_test_opt['wape']:.2f}%  (zero_pred={m_test_opt['zero_pred_rate']:.1f}%)")
print(f"  Gap (Test - Val) = {gap_wape:+.2f}%")

LOWER_BOUND, UPPER_BOUND = 75.0, 76.5
if LOWER_BOUND <= m_test_opt["wape"] <= UPPER_BOUND:
    verdict = "PASS — Model generalizes well (Test WAPE in target range)"
elif m_test_opt["wape"] < LOWER_BOUND:
    verdict = "BETTER THAN EXPECTED — Test WAPE below target"
elif gap_wape <= 3.0:
    verdict = "MARGINAL — Small gap but Test WAPE slightly above target"
else:
    verdict = "FAIL — Overfitting or distribution shift remains"
print(f"\n  Verdict: {verdict}")

preds_test_val_thr = np.where(probs_tuned_test < THRESHOLD_VAL, 0.0, demand_tuned_test)
m_test_val_thr = compute_metrics_full(y_test, preds_test_val_thr)

# Sweep thresholds for reference
sweep_thresholds = np.round(np.arange(0.00, 0.51, 0.01), 2)
sweep_results = []
for thr in sweep_thresholds:
    p = np.where(probs_tuned_test < thr, 0.0, demand_tuned_test)
    w = wape_score(y_test.values, p)
    zpred = (p == 0).mean() * 100
    sweep_results.append({"threshold": thr, "test_wape": w, "zero_pred_pct": zpred})
sweep_df = pd.DataFrame(sweep_results)

print(f"\n  {'Threshold':<15} {'RMSE':>10} {'MAE':>10} {'WAPE':>10} {'Zero Pred%':>12}")
for thr, m, name in [
    (THRESHOLD_OPTUNA, m_test_opt, "Optuna"),
    (THRESHOLD_VAL,    m_test_val_thr, "Val search "),
]:
    print(f"  {name:<15} {m['rmse']:>10.2f} {m['mae']:>10.2f} "
          f"{m['wape']:>9.2f}%  {m['zero_pred_rate']:>10.1f}%")

best_row = sweep_df.loc[sweep_df["test_wape"].idxmin()]
print(f"\n  [Reference] Best threshold on Test: "
      f"{best_row['threshold']:.2f}  (Test WAPE={best_row['test_wape']:.2f}%)")
print(f"  Distance to best: "
      f"0.153 off by {m_test_opt['wape'] - best_row['test_wape']:+.2f}%  |  "
      f"0.05 off by {m_test_val_thr['wape'] - best_row['test_wape']:+.2f}%")


fi_clf = pd.DataFrame({
    "feature": MODEL_FEATURES,
    "gain_clf": clf_tuned.feature_importances_,
}).sort_values("gain_clf", ascending=False).reset_index(drop=True)

fi_reg = pd.DataFrame({
    "feature": MODEL_FEATURES,
    "gain_reg": reg_tuned.feature_importances_,
}).sort_values("gain_reg", ascending=False).reset_index(drop=True)

fi_merged = fi_clf.rename(columns={"gain_clf": "gain"}).copy()
fi_merged["rank_clf"] = fi_merged.index + 1
fi_merged = fi_merged.merge(
    fi_reg[["feature", "gain_reg"]].assign(
        rank_reg=fi_reg.reset_index().index + 1
    ),
    on="feature", how="left"
)
fi_merged["rank_delta"] = fi_merged["rank_reg"] - fi_merged["rank_clf"]

INTERMITTENT_FEATURES = [
    "Days_Since_Last_Sale", "Consecutive_Zeros",
    "Zero_Rate_7d", "Zero_Rate_30d", "Demand_Velocity_7d"
]
TE_FEATURES = [c for c in MODEL_FEATURES if c.endswith("_TE")]

print("\n  Intermittent features rank (Classifier vs Regressor) ")
print(f"  {'Feature':<30} {'Rank CLF':>10} {'Rank REG':>10} {'Gain CLF':>12} {'Gain REG':>12}")

for feat in INTERMITTENT_FEATURES:
    row = fi_merged[fi_merged["feature"] == feat]
    if row.empty:
        print(f"  {feat:<30}  NOT FOUND")
        continue
    r = row.iloc[0]
    print(f"  {feat:<30} {int(r['rank_clf']):>10} {int(r['rank_reg']):>10} "
          f"{r['gain']:>12.1f} {r['gain_reg']:>12.1f}")

print(f"  {'Feature':<30} {'Rank CLF':>10} {'Rank REG':>10} {'Gain CLF':>12} {'Gain REG':>12}")
for feat in TE_FEATURES:
    row = fi_merged[fi_merged["feature"] == feat]
    if row.empty:
        continue
    r = row.iloc[0]
    flag = " HIGH" if (r["rank_clf"] <= 5 or r["rank_reg"] <= 5) else ""
    print(f"  {feat:<30} {int(r['rank_clf']):>10} {int(r['rank_reg']):>10} "
          f"{r['gain']:>12.1f} {r['gain_reg']:>12.1f}  {flag}")


# Visualization
PALETTE = {
    "bg":       "#0f1117",
    "panel":    "#1a1d27",
    "accent1":  "#4f9cf9",
    "accent2":  "#43d9ad",
    "accent3":  "#ff6b6b",
    "accent4":  "#ffd93d",
    "accent5":  "#c77dff",
    "text":     "#e8eaf0",
    "subtext":  "#8b8fa8",
    "grid":     "#2a2d3a",
}

matplotlib.rcParams.update({
    "figure.facecolor":  PALETTE["bg"],
    "axes.facecolor":    PALETTE["panel"],
    "axes.edgecolor":    PALETTE["grid"],
    "axes.labelcolor":   PALETTE["text"],
    "xtick.color":       PALETTE["subtext"],
    "ytick.color":       PALETTE["subtext"],
    "text.color":        PALETTE["text"],
    "grid.color":        PALETTE["grid"],
    "grid.linestyle":    "--",
    "grid.alpha":        0.5,
    "font.family":       "monospace",
})

fig = plt.figure(figsize=(22, 16), facecolor=PALETTE["bg"])
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.38,
                       left=0.06, right=0.97, top=0.92, bottom=0.06)

ax_gap    = fig.add_subplot(gs[0, 0])
ax_sweep  = fig.add_subplot(gs[0, 1])
ax_thr2   = fig.add_subplot(gs[0, 2])
ax_clf_fi = fig.add_subplot(gs[1, 0])
ax_reg_fi = fig.add_subplot(gs[1, 1])
ax_rank   = fig.add_subplot(gs[1, 2])

fig.suptitle(
    "Deep Analysis v3  |  Hybrid Tuned · Threshold · Feature Importance",
    fontsize=15, color=PALETTE["text"], fontweight="bold", y=0.97
)

# Panel 1: Val/Test Gap
splits   = ["Val (Optuna)", "Test (thr=0.153)", "Test (thr=0.05)"]
wape_vals = [m_val_opt["wape"], m_test_opt["wape"], m_test_val_thr["wape"]]
bar_colors = [PALETTE["accent2"], PALETTE["accent1"], PALETTE["accent4"]]
bars = ax_gap.bar(splits, wape_vals, color=bar_colors, width=0.55,
                  edgecolor=PALETTE["bg"], linewidth=1.5)

ax_gap.axhspan(75.0, 76.5, alpha=0.12, color=PALETTE["accent2"],
               label="Target zone [75–76.5%]")
ax_gap.axhline(VAL_WAPE_REPORTED, color=PALETTE["accent2"],
               linestyle="--", linewidth=1.2, alpha=0.7, label=f"Reported Val {VAL_WAPE_REPORTED}%")

for bar, val in zip(bars, wape_vals):
    ax_gap.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.2,
                f"{val:.2f}%", ha="center", va="bottom",
                color=PALETTE["text"], fontsize=9.5, fontweight="bold")

ax_gap.set_title("TASK 1 — Val/Test Gap", color=PALETTE["text"], fontsize=11, pad=10)
ax_gap.set_ylabel("WAPE (%)", color=PALETTE["subtext"])
ax_gap.set_ylim(min(wape_vals) - 3, max(wape_vals) + 4)
ax_gap.legend(fontsize=7.5, facecolor=PALETTE["panel"],
              edgecolor=PALETTE["grid"], labelcolor=PALETTE["text"])
ax_gap.grid(axis="y")
ax_gap.tick_params(axis="x", labelsize=8)

verdict_short = "GENERALIZED" if LOWER_BOUND <= m_test_opt["wape"] <= UPPER_BOUND \
    else ("BETTER" if m_test_opt["wape"] < LOWER_BOUND else "CHECK")
ax_gap.text(0.5, 0.04, verdict_short, transform=ax_gap.transAxes,
            ha="center", fontsize=8.5, color=PALETTE["accent2"],
            fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", facecolor=PALETTE["panel"],
                      edgecolor=PALETTE["accent2"], alpha=0.8))

# Panel 2: Threshold Sweep
ax_sweep.plot(sweep_df["threshold"], sweep_df["test_wape"],
              color=PALETTE["accent1"], linewidth=2.0, label="Test WAPE")

ax2_right = ax_sweep.twinx()
ax2_right.plot(sweep_df["threshold"], sweep_df["zero_pred_pct"],
               color=PALETTE["accent5"], linewidth=1.4, linestyle=":",
               alpha=0.7, label="Zero Pred %")
ax2_right.set_ylabel("Zero Pred %", color=PALETTE["accent5"], fontsize=8)
ax2_right.tick_params(colors=PALETTE["accent5"])
ax2_right.spines["right"].set_edgecolor(PALETTE["accent5"])

for thr, col, name in [
    (THRESHOLD_OPTUNA, PALETTE["accent4"], f"Optuna ({THRESHOLD_OPTUNA})"),
    (THRESHOLD_VAL,    PALETTE["accent3"], f"Val search ({THRESHOLD_VAL})"),
]:
    ax_sweep.axvline(thr, color=col, linestyle="--", linewidth=1.5, alpha=0.9)
    wape_at_thr = sweep_df.loc[sweep_df["threshold"] == round(thr, 2), "test_wape"].values
    if len(wape_at_thr):
        ax_sweep.annotate(f"{name}\n{wape_at_thr[0]:.2f}%",
                          xy=(thr, wape_at_thr[0]),
                          xytext=(thr + 0.03, wape_at_thr[0] + 0.6),
                          fontsize=7, color=col,
                          arrowprops=dict(arrowstyle="->", color=col, lw=1.0))

ax_sweep.set_title("TASK 2 — Threshold Sweep (Test Set)", color=PALETTE["text"],
                   fontsize=11, pad=10)
ax_sweep.set_xlabel("Threshold", color=PALETTE["subtext"])
ax_sweep.set_ylabel("WAPE (%)", color=PALETTE["subtext"])
ax_sweep.grid(True)
lines1, labels1 = ax_sweep.get_legend_handles_labels()
lines2, labels2 = ax2_right.get_legend_handles_labels()
ax_sweep.legend(lines1 + lines2, labels1 + labels2,
                fontsize=7.5, facecolor=PALETTE["panel"],
                edgecolor=PALETTE["grid"], labelcolor=PALETTE["text"])

# Panel 3: Two-threshold head-to-head
metrics_two = {
    "0.153\n(Optuna)":     m_test_opt,
    "0.05\n(Val search)": m_test_val_thr,
}
metric_keys = ["wape", "mae", "rmse", "zero_pred_rate"]
metric_labels = ["WAPE (%)", "MAE", "RMSE", "Zero Pred %"]

x_pos = np.arange(len(metric_keys))
width = 0.3
colors_two = [PALETTE["accent4"], PALETTE["accent3"]]

for i, (name, m) in enumerate(metrics_two.items()):
    vals_normalized = [
        m["wape"],
        m["mae"] / 100,
        m["rmse"] / 1000,
        m["zero_pred_rate"],
    ]
    bars_two = ax_thr2.bar(x_pos + i * width, vals_normalized,
                           width, label=name, color=colors_two[i],
                           edgecolor=PALETTE["bg"], alpha=0.85)
    for bar, raw_val in zip(bars_two, [m["wape"], m["mae"], m["rmse"], m["zero_pred_rate"]]):
        ax_thr2.text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 0.1,
                     f"{raw_val:.1f}", ha="center", va="bottom",
                     color=PALETTE["text"], fontsize=7)

ax_thr2.set_xticks(x_pos + width / 2)
ax_thr2.set_xticklabels(["WAPE\n(%)", "MAE\n(÷100)", "RMSE\n(÷1000)", "ZeroPred\n(%)"],
                         fontsize=7.5)
ax_thr2.set_title("TASK 2 — Threshold Head-to-Head (Test)", color=PALETTE["text"],
                  fontsize=11, pad=10)
ax_thr2.legend(fontsize=8, facecolor=PALETTE["panel"],
               edgecolor=PALETTE["grid"], labelcolor=PALETTE["text"])
ax_thr2.grid(axis="y")

# Panel 4: Classifier Feature Importance
top_n = 20
top_clf = fi_clf.head(top_n)

colors_clf = []
for feat in top_clf["feature"]:
    if feat in INTERMITTENT_FEATURES:
        colors_clf.append(PALETTE["accent2"])
    elif feat in TE_FEATURES:
        colors_clf.append(PALETTE["accent4"])
    else:
        colors_clf.append(PALETTE["accent1"])

ax_clf_fi.barh(top_clf["feature"][::-1], top_clf["gain_clf"][::-1],
               color=colors_clf[::-1], edgecolor=PALETTE["bg"], linewidth=0.5)
ax_clf_fi.set_title("TASK 3 — Classifier FI (Stage 1)\n"
                    "Intermittent  TargetEnc  Other",
                    color=PALETTE["text"], fontsize=9.5, pad=8)
ax_clf_fi.set_xlabel("Gain", color=PALETTE["subtext"], fontsize=8)
ax_clf_fi.tick_params(axis="y", labelsize=7)
ax_clf_fi.grid(axis="x")

from matplotlib.patches import Patch
legend_els_clf = [
    Patch(facecolor=PALETTE["accent2"], label="Intermittent"),
    Patch(facecolor=PALETTE["accent4"], label="TargetEnc (OOF)"),
    Patch(facecolor=PALETTE["accent1"], label="Other"),
]
ax_clf_fi.legend(handles=legend_els_clf, fontsize=7,
                 facecolor=PALETTE["panel"], edgecolor=PALETTE["grid"],
                 labelcolor=PALETTE["text"], loc="lower right")

# Panel 5: Regressor Feature Importance
top_reg = fi_reg.head(top_n)
colors_reg = []
for feat in top_reg["feature"]:
    if feat in INTERMITTENT_FEATURES:
        colors_reg.append(PALETTE["accent2"])
    elif feat in TE_FEATURES:
        colors_reg.append(PALETTE["accent4"])
    else:
        colors_reg.append(PALETTE["accent1"])

ax_reg_fi.barh(top_reg["feature"][::-1], top_reg["gain_reg"][::-1],
               color=colors_reg[::-1], edgecolor=PALETTE["bg"], linewidth=0.5)
ax_reg_fi.set_title("TASK 3 — Regressor FI (Stage 2)\n"
                    "Intermittent  TargetEnc  Other",
                    color=PALETTE["text"], fontsize=9.5, pad=8)
ax_reg_fi.set_xlabel("Gain", color=PALETTE["subtext"], fontsize=8)
ax_reg_fi.tick_params(axis="y", labelsize=7)
ax_reg_fi.grid(axis="x")
ax_reg_fi.legend(handles=legend_els_clf, fontsize=7,
                 facecolor=PALETTE["panel"], edgecolor=PALETTE["grid"],
                 labelcolor=PALETTE["text"], loc="lower right")

# Panel 6: Rank Delta Scatter
common_feats = fi_merged.dropna(subset=["rank_clf", "rank_reg"]).head(30)
sc_colors = [
    PALETTE["accent2"] if f in INTERMITTENT_FEATURES
    else (PALETTE["accent4"] if f in TE_FEATURES else PALETTE["subtext"])
    for f in common_feats["feature"]
]
ax_rank.scatter(common_feats["rank_clf"], common_feats["rank_reg"],
                c=sc_colors, s=60, edgecolors=PALETTE["bg"], linewidths=0.5, zorder=3)
ax_rank.plot([1, 30], [1, 30], color=PALETTE["grid"], linestyle="--",
             linewidth=1.0, alpha=0.6, label="rank_clf = rank_reg")

for _, row_r in common_feats.iterrows():
    if row_r["feature"] in INTERMITTENT_FEATURES:
        ax_rank.annotate(
            row_r["feature"].replace("_", "\n"),
            xy=(row_r["rank_clf"], row_r["rank_reg"]),
            fontsize=6.5, color=PALETTE["accent2"],
            xytext=(3, 3), textcoords="offset points",
        )

ax_rank.set_xlabel("Rank in Classifier", color=PALETTE["subtext"], fontsize=8)
ax_rank.set_ylabel("Rank in Regressor", color=PALETTE["subtext"], fontsize=8)
ax_rank.set_title("TASK 3 — Rank Shift: CLF vs REG\n"
                  "Intermittent  TargetEnc  Other",
                  color=PALETTE["text"], fontsize=9.5, pad=8)
ax_rank.grid(True)
ax_rank.invert_yaxis()
ax_rank.invert_xaxis()
legend_els_rank = [
    Patch(facecolor=PALETTE["accent2"], label="Intermittent"),
    Patch(facecolor=PALETTE["accent4"], label="TargetEnc (OOF)"),
    Patch(facecolor=PALETTE["subtext"],  label="Other"),
]
ax_rank.legend(handles=legend_els_rank, fontsize=7,
               facecolor=PALETTE["panel"], edgecolor=PALETTE["grid"],
               labelcolor=PALETTE["text"])

plt.savefig("analysis_v3_deep.png", dpi=160, bbox_inches="tight",
            facecolor=PALETTE["bg"])
plt.close()

print("SUMMARY REPORT")

print(f"""
  TASK 1: Generalization Check
  Val  WAPE  = {m_val_opt['wape']:6.2f}%  (threshold = {THRESHOLD_OPTUNA})
  Test WAPE  = {m_test_opt['wape']:6.2f}%  (threshold = {THRESHOLD_OPTUNA})
  Gap        = {gap_wape:+6.2f}%
  Verdict    : {verdict_short}

  TASK 2: Threshold Comparison (Test Set)
  thr=0.153  WAPE={m_test_opt['wape']:6.2f}%   ZeroPred={m_test_opt['zero_pred_rate']:5.1f}%
  thr=0.05   WAPE={m_test_val_thr['wape']:6.2f}%   ZeroPred={m_test_val_thr['zero_pred_rate']:5.1f}%
  Winner     : {"0.153 (Optuna)" if m_test_opt['wape'] < m_test_val_thr['wape'] else "0.05 (Val search)"}
  Delta WAPE : {abs(m_test_opt['wape'] - m_test_val_thr['wape']):.2f}pp

  TASK 3: Feature Importance after Leakage Fix
  Classifier top intermittent features:
""")

for feat in INTERMITTENT_FEATURES:
    row = fi_merged[fi_merged["feature"] == feat]
    if not row.empty:
        r = row.iloc[0]
        print(f"    {feat:<30}  rank_clf={int(r['rank_clf']):3d}  rank_reg={int(r['rank_reg']):3d}")

print("""
  TargetEnc rank check (OOF — should be lower than raw):
""")
for feat in TE_FEATURES:
    row = fi_merged[fi_merged["feature"] == feat]
    if not row.empty:
        r = row.iloc[0]
        warn = " HIGH" if (r["rank_clf"] <= 3 or r["rank_reg"] <= 3) else " OK"
        print(f"    {feat:<30}  rank_clf={int(r['rank_clf']):3d}  rank_reg={int(r['rank_reg']):3d}{warn}")


  THRESHOLD_OPTUNA (from saved model) : 0.0734
  THRESHOLD_VAL    (recomputed sweep) : 0.0500  (Val WAPE=75.07%)

  [Threshold = 0.0734340234995807]
  Val  WAPE = 75.07%  (zero_pred=52.9%)
  Test WAPE = 72.60%  (zero_pred=51.2%)
  Gap (Test - Val) = -2.47%

  Verdict: BETTER THAN EXPECTED — Test WAPE below target

  Threshold             RMSE        MAE       WAPE   Zero Pred%
  Optuna            20043.25    1518.60     72.60%        51.2%
  Val search        20043.25    1518.60     72.60%        51.2%

  [Reference] Best threshold on Test: 0.01  (Test WAPE=72.60%)
  Distance to best: 0.153 off by +0.00%  |  0.05 off by +0.00%

  Intermittent features rank (Classifier vs Regressor) 
  Feature                          Rank CLF   Rank REG     Gain CLF     Gain REG
  Days_Since_Last_Sale                    9         22          8.0        261.0
  Consecutive_Zeros                      65         53          0.0         16.0
  Zero_Rate_7d                            3          6         72